# 🦟 Plasmodium falciparum Genomic Variant Pipeline
## Molecular Surveillance for Antimalarial Drug Resistance

---

**Laboratory Scientist Training & Demonstration Notebook**

This notebook demonstrates a complete bioinformatics workflow for detecting drug resistance mutations in *Plasmodium falciparum* malaria parasites from next-generation sequencing data.

### Learning Objectives
By the end of this notebook, you will understand how to:
1. Perform quality control on raw FASTQ sequencing data
2. Align reads to the *P. falciparum* 3D7 reference genome
3. Call genetic variants (SNPs and INDELs)
4. Annotate variants with functional information
5. Detect antimalarial drug resistance markers
6. Store and query results in a database
7. Visualize resistance patterns for surveillance

### Target Genes for Drug Resistance
| Gene | Drug(s) Affected | Key Mutations |
|------|------------------|---------------|
| *pfcrt* | Chloroquine | K76T |
| *pfmdr1* | Lumefantrine, Mefloquine | N86Y, Y184F, D1246Y |
| *kelch13* | Artemisinin | C580Y, R539T, Y493H |
| *dhfr* | Pyrimethamine | N51I, C59R, S108N |
| *dhps* | Sulfadoxine | A437G, K540E, A581G |

---

## 🔧 Setup and Imports

First, let's import all required modules and configure our environment.

In [ ]:
# Standard library imports
import sys
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add source directory to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Data science imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11

# Pipeline module imports
from preprocessing import FASTQProcessor, process_fastq
from alignment import ReadAligner, align_fastq_to_reference
from variant_calling import VariantCaller, call_variants_from_alignment, KNOWN_RESISTANCE_MUTATIONS
from annotation import VariantAnnotator, annotate_vcf_file
from resistance_detection import ResistanceDetector, detect_resistance, DRUG_DATABASE
from database import DatabaseManager, create_database
from utils import setup_logging, ensure_directory

# Setup logging
logger = setup_logging('pipeline_demo')

print("✅ All modules loaded successfully!")
print(f"📁 Project root: {PROJECT_ROOT}")

In [ ]:
# Define project paths
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATABASE_DIR = PROJECT_ROOT / 'database'
REPORTS_DIR = PROJECT_ROOT / 'reports'

# Ensure directories exist
for dir_path in [DATA_RAW, DATA_PROCESSED, DATABASE_DIR, REPORTS_DIR]:
    ensure_directory(dir_path)

# Define sample information
SAMPLE_ID = "MWI-2024-001"
SAMPLE_METADATA = {
    'sample_id': SAMPLE_ID,
    'collection_date': '2024-03-15',
    'location': 'Blantyre District Hospital',
    'country': 'Malawi',
    'region': 'Southern Region',
    'patient_age': 8,
    'parasitemia': 2.5,  # percentage
    'sequencing_platform': 'Illumina MiSeq',
    'read_length': 150
}

print("📊 Sample Metadata:")
for key, value in SAMPLE_METADATA.items():
    print(f"   {key}: {value}")

---

## 📖 Step 1: FASTQ Quality Control

### Background
FASTQ files contain raw sequencing reads with quality scores. Before analysis, we must:
- Assess overall data quality
- Filter out low-quality reads
- Trim adapter sequences and low-quality bases

Quality scores use the Phred scale: Q20 = 1% error rate, Q30 = 0.1% error rate.

In [ ]:
# Load the demo FASTQ file
fastq_file = DATA_RAW / 'demo_sample.fastq'

if not fastq_file.exists():
    print("⚠️  Demo FASTQ not found. Generating synthetic data...")
    # Generate synthetic FASTQ if not present
    from preprocessing import FASTQProcessor
    # This would generate synthetic data - see run_pipeline.py for generation code
else:
    print(f"✅ Found FASTQ file: {fastq_file}")
    print(f"   File size: {fastq_file.stat().st_size / 1024:.1f} KB")

In [ ]:
# Initialize FASTQ processor
processor = FASTQProcessor(str(fastq_file))

# Calculate quality statistics
print("🔬 Analyzing FASTQ quality metrics...\n")
qc_stats = processor.get_quality_stats()

# Display QC summary
print("="*60)
print("           FASTQ QUALITY CONTROL REPORT")
print("="*60)
print(f"\n📊 Read Statistics:")
print(f"   Total reads:        {qc_stats.total_reads:,}")
print(f"   Total bases:        {qc_stats.total_bases:,}")
print(f"   Mean read length:   {qc_stats.mean_length:.1f} bp")
print(f"   Length range:       {qc_stats.min_length} - {qc_stats.max_length} bp")

print(f"\n🎯 Quality Metrics:")
print(f"   Mean quality score: {qc_stats.mean_quality:.1f}")
print(f"   Q20 (≥99% acc):     {qc_stats.q20_percentage:.1f}%")
print(f"   Q30 (≥99.9% acc):   {qc_stats.q30_percentage:.1f}%")

print(f"\n🧬 Sequence Composition:")
print(f"   GC content:         {qc_stats.gc_content:.1f}%")
print(f"   N content:          {qc_stats.n_content:.2f}%")
print("="*60)

# Quality assessment
if qc_stats.q30_percentage >= 80:
    print("\n✅ PASS: Excellent quality data suitable for variant calling")
elif qc_stats.q30_percentage >= 60:
    print("\n⚠️  WARNING: Acceptable quality, consider additional filtering")
else:
    print("\n❌ FAIL: Poor quality data, re-sequencing recommended")

In [ ]:
# Visualize quality metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Per-position quality boxplot (simulated)
ax1 = axes[0, 0]
positions = list(qc_stats.per_position_quality.keys())[:50]  # First 50 positions
qualities = [qc_stats.per_position_quality[p] for p in positions]
ax1.bar(positions, qualities, color='steelblue', alpha=0.7)
ax1.axhline(y=30, color='green', linestyle='--', label='Q30 threshold')
ax1.axhline(y=20, color='orange', linestyle='--', label='Q20 threshold')
ax1.set_xlabel('Position in Read (bp)')
ax1.set_ylabel('Mean Quality Score')
ax1.set_title('Per-Position Quality Score')
ax1.legend()
ax1.set_ylim(0, 42)

# 2. Quality score distribution
ax2 = axes[0, 1]
quality_dist = np.random.normal(qc_stats.mean_quality, 3, 1000)
quality_dist = quality_dist[(quality_dist >= 0) & (quality_dist <= 41)]
ax2.hist(quality_dist, bins=40, color='teal', alpha=0.7, edgecolor='black')
ax2.axvline(x=30, color='green', linestyle='--', linewidth=2, label='Q30')
ax2.axvline(x=20, color='orange', linestyle='--', linewidth=2, label='Q20')
ax2.axvline(x=qc_stats.mean_quality, color='red', linestyle='-', linewidth=2, label=f'Mean ({qc_stats.mean_quality:.1f})')
ax2.set_xlabel('Quality Score (Phred)')
ax2.set_ylabel('Frequency')
ax2.set_title('Quality Score Distribution')
ax2.legend()

# 3. Read length distribution
ax3 = axes[1, 0]
length_dist = np.random.normal(qc_stats.mean_length, 10, qc_stats.total_reads)
length_dist = length_dist[(length_dist >= qc_stats.min_length) & (length_dist <= qc_stats.max_length)]
ax3.hist(length_dist, bins=30, color='coral', alpha=0.7, edgecolor='black')
ax3.axvline(x=qc_stats.mean_length, color='red', linestyle='-', linewidth=2, label=f'Mean ({qc_stats.mean_length:.0f} bp)')
ax3.set_xlabel('Read Length (bp)')
ax3.set_ylabel('Frequency')
ax3.set_title('Read Length Distribution')
ax3.legend()

# 4. GC content vs P. falciparum expected
ax4 = axes[1, 1]
categories = ['Sample GC%', 'P. falciparum Expected\n(~19-20%)', 'Human (Control)\n(~41%)']
values = [qc_stats.gc_content, 19.5, 41]
colors = ['steelblue', 'forestgreen', 'gray']
bars = ax4.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
ax4.set_ylabel('GC Content (%)')
ax4.set_title('GC Content Comparison')
ax4.set_ylim(0, 60)
for bar, val in zip(bars, values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'qc_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 QC plot saved to: {REPORTS_DIR / 'qc_metrics.png'}")

In [ ]:
# Filter reads based on quality thresholds
print("🔧 Filtering reads...")
print(f"   Minimum quality: Q20")
print(f"   Minimum length:  50 bp")
print(f"   Maximum N%:      10%")

filtered_fastq = DATA_PROCESSED / f'{SAMPLE_ID}_filtered.fastq'
filter_stats = processor.filter_reads(
    output_path=str(filtered_fastq),
    min_quality=20,
    min_length=50,
    max_n_fraction=0.1
)

print(f"\n📈 Filtering Results:")
print(f"   Reads before:  {filter_stats['reads_before']:,}")
print(f"   Reads after:   {filter_stats['reads_after']:,}")
print(f"   Reads removed: {filter_stats['reads_removed']:,} ({filter_stats['percent_removed']:.1f}%)")
print(f"   Bases before:  {filter_stats['bases_before']:,}")
print(f"   Bases after:   {filter_stats['bases_after']:,}")

print(f"\n✅ Filtered FASTQ saved: {filtered_fastq}")

---

## 🧬 Step 2: Read Alignment

### Background
Reads are aligned to the *P. falciparum* 3D7 reference genome (PlasmoDB). This identifies where each read originates in the genome.

**In production**, we would use:
- **BWA-MEM** or **Bowtie2** for alignment
- **SAMtools** for BAM processing

For this demonstration, we simulate the alignment process to show the workflow.

In [ ]:
# Initialize aligner
aligner = ReadAligner(reference_name="Pf3D7")

print("🧬 Reference Genome: Plasmodium falciparum 3D7")
print("\n📍 Target genes for resistance marker detection:")
print("="*55)

for gene, info in aligner.reference.GENE_COORDINATES.items():
    print(f"   {gene:10} | Chr {info['chromosome']:2} | {info['start']:>10,} - {info['end']:>10,}")
print("="*55)

In [ ]:
# Perform alignment
print("🔄 Aligning reads to reference genome...\n")

sam_file = DATA_PROCESSED / f'{SAMPLE_ID}.sam'
alignment_stats = aligner.align_reads(
    fastq_path=str(filtered_fastq),
    output_path=str(sam_file),
    sample_id=SAMPLE_ID
)

print("="*60)
print("              ALIGNMENT STATISTICS")
print("="*60)
print(f"\n📊 Mapping Summary:")
print(f"   Total reads:      {alignment_stats.total_reads:,}")
print(f"   Mapped reads:     {alignment_stats.mapped_reads:,} ({alignment_stats.mapping_rate:.1f}%)")
print(f"   Unmapped reads:   {alignment_stats.unmapped_reads:,}")

print(f"\n🎯 Mapping Quality:")
print(f"   Mean MAPQ:        {alignment_stats.mean_mapq:.1f}")
print(f"   Unique mappings:  {alignment_stats.unique_mappings:,}")
print(f"   Multi-mappings:   {alignment_stats.multi_mappings:,}")

print(f"\n📍 Reads in Target Genes:")
for gene, count in sorted(alignment_stats.reads_per_chromosome.items(), key=lambda x: -x[1]):
    pct = (count / alignment_stats.mapped_reads) * 100
    bar = '█' * int(pct / 2)
    print(f"   {gene:10} {count:5,} reads ({pct:5.1f}%) {bar}")

print("="*60)
print(f"\n✅ SAM file saved: {sam_file}")

In [ ]:
# Visualize alignment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Mapping status pie chart
ax1 = axes[0]
mapping_data = [alignment_stats.mapped_reads, alignment_stats.unmapped_reads]
labels = ['Mapped', 'Unmapped']
colors = ['#2ecc71', '#e74c3c']
explode = (0.02, 0)
ax1.pie(mapping_data, labels=labels, colors=colors, autopct='%1.1f%%', 
        explode=explode, shadow=True, startangle=90)
ax1.set_title(f'Read Mapping Status\n(n={alignment_stats.total_reads:,} reads)', fontsize=12)

# 2. Reads per gene bar chart
ax2 = axes[1]
genes = list(alignment_stats.reads_per_chromosome.keys())
counts = list(alignment_stats.reads_per_chromosome.values())
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(genes)))
bars = ax2.barh(genes, counts, color=colors, edgecolor='black', alpha=0.8)
ax2.set_xlabel('Number of Mapped Reads')
ax2.set_title('Read Distribution Across Target Genes', fontsize=12)
ax2.invert_yaxis()

# Add value labels
for bar, count in zip(bars, counts):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
             f'{count:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'alignment_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Alignment plot saved to: {REPORTS_DIR / 'alignment_stats.png'}")

---

## 🔬 Step 3: Variant Calling

### Background
Variant calling identifies differences between the sample and reference genome:
- **SNPs** (Single Nucleotide Polymorphisms): Single base changes
- **INDELs**: Insertions or deletions of bases

Output is in **VCF** (Variant Call Format), the standard format for genomic variants.

**In production**, we would use:
- **GATK HaplotypeCaller** or **FreeBayes** for variant calling
- Quality filtering with GATK VQSR or hard filters

In [ ]:
# Initialize variant caller
caller = VariantCaller(reference_name="Pf3D7")

# Display known resistance mutations we're looking for
print("🎯 Known Resistance Mutations in Database:")
print("="*70)

for gene, mutations in KNOWN_RESISTANCE_MUTATIONS.items():
    print(f"\n📍 {gene}:")
    for mutation, info in mutations.items():
        drugs = ', '.join(info['drugs'])
        print(f"   • {mutation:10} → {drugs}")

print("\n" + "="*70)

In [ ]:
# Call variants from alignment
print("🔬 Calling variants from alignment...\n")

vcf_file = DATA_PROCESSED / f'{SAMPLE_ID}.vcf'
variant_stats = caller.call_variants(
    sam_path=str(sam_file),
    output_path=str(vcf_file),
    sample_id=SAMPLE_ID,
    min_depth=10,
    min_quality=20
)

print("="*60)
print("              VARIANT CALLING RESULTS")
print("="*60)
print(f"\n📊 Variant Summary:")
print(f"   Total variants:     {variant_stats.total_variants:,}")
print(f"   SNPs:               {variant_stats.snps:,}")
print(f"   Insertions:         {variant_stats.insertions:,}")
print(f"   Deletions:          {variant_stats.deletions:,}")

print(f"\n🧬 Variants by Gene:")
for gene, count in sorted(variant_stats.variants_per_gene.items(), key=lambda x: -x[1]):
    bar = '█' * min(count, 20)
    print(f"   {gene:10} {count:3} variants {bar}")

print(f"\n🎯 Quality Metrics:")
print(f"   Mean quality:       {variant_stats.mean_quality:.1f}")
print(f"   Mean depth:         {variant_stats.mean_depth:.1f}x")
print(f"   Ti/Tv ratio:        {variant_stats.ti_tv_ratio:.2f}")

print("="*60)
print(f"\n✅ VCF file saved: {vcf_file}")

In [ ]:
# Preview VCF file
print("📄 VCF File Preview (first 20 lines):")
print("="*80)
with open(vcf_file, 'r') as f:
    for i, line in enumerate(f):
        if i < 20:
            print(line.rstrip())
        else:
            break
print("...")
print("="*80)

In [ ]:
# Visualize variant calling results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Variant type distribution
ax1 = axes[0, 0]
var_types = ['SNPs', 'Insertions', 'Deletions']
var_counts = [variant_stats.snps, variant_stats.insertions, variant_stats.deletions]
colors = ['#3498db', '#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax1.pie(var_counts, labels=var_types, colors=colors, 
                                    autopct='%1.1f%%', startangle=90, explode=(0.02, 0, 0))
ax1.set_title(f'Variant Type Distribution\n(n={variant_stats.total_variants} variants)', fontsize=12)

# 2. Variants per gene
ax2 = axes[0, 1]
genes = list(variant_stats.variants_per_gene.keys())
counts = list(variant_stats.variants_per_gene.values())
colors = plt.cm.Set2(range(len(genes)))
bars = ax2.bar(genes, counts, color=colors, edgecolor='black', alpha=0.8)
ax2.set_xlabel('Gene')
ax2.set_ylabel('Number of Variants')
ax2.set_title('Variants per Target Gene', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
for bar, count in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
             str(count), ha='center', va='bottom', fontweight='bold')

# 3. Quality score distribution (simulated)
ax3 = axes[1, 0]
qual_dist = np.random.gamma(shape=5, scale=variant_stats.mean_quality/5, size=variant_stats.total_variants)
ax3.hist(qual_dist, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax3.axvline(x=30, color='red', linestyle='--', linewidth=2, label='QUAL ≥ 30 threshold')
ax3.set_xlabel('Variant Quality Score')
ax3.set_ylabel('Frequency')
ax3.set_title('Variant Quality Distribution', fontsize=12)
ax3.legend()

# 4. Read depth distribution (simulated)
ax4 = axes[1, 1]
depth_dist = np.random.poisson(lam=variant_stats.mean_depth, size=variant_stats.total_variants)
ax4.hist(depth_dist, bins=30, color='teal', alpha=0.7, edgecolor='black')
ax4.axvline(x=10, color='red', linestyle='--', linewidth=2, label='DP ≥ 10 threshold')
ax4.axvline(x=variant_stats.mean_depth, color='orange', linestyle='-', linewidth=2, 
            label=f'Mean ({variant_stats.mean_depth:.0f}x)')
ax4.set_xlabel('Read Depth')
ax4.set_ylabel('Frequency')
ax4.set_title('Read Depth Distribution', fontsize=12)
ax4.legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'variant_calling.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Variant calling plot saved to: {REPORTS_DIR / 'variant_calling.png'}")

---

## 🏷️ Step 4: Variant Annotation

### Background
Annotation adds biological context to variants:
- **Gene name**: Which gene is affected
- **Effect**: Missense, synonymous, nonsense, frameshift
- **Amino acid change**: e.g., K76T (Lysine → Threonine at position 76)
- **Resistance marker**: Is this a known drug resistance mutation?

**In production**, we would use:
- **SnpEff** or **VEP** for comprehensive annotation
- Custom databases for resistance markers

In [ ]:
# Initialize annotator
annotator = VariantAnnotator()

# Annotate variants
print("🏷️ Annotating variants with functional information...\n")

annotated_vcf = DATA_PROCESSED / f'{SAMPLE_ID}_annotated.vcf'
annotation_stats = annotator.annotate_vcf(
    vcf_path=str(vcf_file),
    output_path=str(annotated_vcf)
)

print("="*60)
print("              VARIANT ANNOTATION RESULTS")
print("="*60)
print(f"\n📊 Annotation Summary:")
print(f"   Total annotated:      {annotation_stats.total_annotated:,}")
print(f"   In known genes:       {annotation_stats.in_gene:,}")
print(f"   Intergenic:           {annotation_stats.intergenic:,}")

print(f"\n🧬 Functional Effects:")
print(f"   Missense (AA change): {annotation_stats.missense:,}")
print(f"   Synonymous (silent):  {annotation_stats.synonymous:,}")
print(f"   Nonsense (stop):      {annotation_stats.nonsense:,}")
print(f"   Frameshift:           {annotation_stats.frameshift:,}")

print(f"\n🎯 Resistance Markers Found: {annotation_stats.resistance_mutations}")
print("="*60)

print(f"\n✅ Annotated VCF saved: {annotated_vcf}")

In [ ]:
# Show gene-level summary
print("\n📍 Per-Gene Variant Summary:")
print("="*70)

gene_summary = annotator.get_gene_summary(str(annotated_vcf))

summary_data = []
for gene, stats in gene_summary.items():
    summary_data.append({
        'Gene': gene,
        'Total': stats['total'],
        'Missense': stats['missense'],
        'Synonymous': stats['synonymous'],
        'Resistance': stats['resistance']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Resistance', ascending=False)
print(summary_df.to_string(index=False))
print("="*70)

In [ ]:
# Visualize annotation results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Functional effect distribution
ax1 = axes[0]
effects = ['Missense', 'Synonymous', 'Nonsense', 'Frameshift']
effect_counts = [annotation_stats.missense, annotation_stats.synonymous, 
                 annotation_stats.nonsense, annotation_stats.frameshift]
colors = ['#e74c3c', '#3498db', '#9b59b6', '#f39c12']
bars = ax1.bar(effects, effect_counts, color=colors, edgecolor='black', alpha=0.8)
ax1.set_ylabel('Number of Variants')
ax1.set_title('Functional Effect Distribution', fontsize=12)
for bar, count in zip(bars, effect_counts):
    if count > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
                 str(count), ha='center', va='bottom', fontweight='bold')

# 2. Gene distribution with resistance highlighting
ax2 = axes[1]
genes = summary_df['Gene'].tolist()
total_counts = summary_df['Total'].tolist()
resistance_counts = summary_df['Resistance'].tolist()

x = np.arange(len(genes))
width = 0.4
bars1 = ax2.bar(x - width/2, total_counts, width, label='Total Variants', color='steelblue', alpha=0.7)
bars2 = ax2.bar(x + width/2, resistance_counts, width, label='Resistance Markers', color='crimson', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(genes, rotation=45, ha='right')
ax2.set_ylabel('Count')
ax2.set_title('Variants by Gene', fontsize=12)
ax2.legend()

# 3. Resistance vs non-resistance pie
ax3 = axes[2]
resistance_total = annotation_stats.resistance_mutations
non_resistance = annotation_stats.total_annotated - resistance_total
pie_data = [resistance_total, non_resistance]
pie_labels = [f'Resistance Markers\n(n={resistance_total})', f'Other Variants\n(n={non_resistance})']
pie_colors = ['#c0392b', '#27ae60']
ax3.pie(pie_data, labels=pie_labels, colors=pie_colors, autopct='%1.1f%%', 
        startangle=90, explode=(0.05, 0))
ax3.set_title('Resistance Marker Proportion', fontsize=12)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'annotation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Annotation plot saved to: {REPORTS_DIR / 'annotation_results.png'}")

---

## 💊 Step 5: Drug Resistance Detection

### Background
This is the critical step for molecular surveillance! We classify the sample's resistance profile based on detected mutations.

**Classification Logic:**
- **RESISTANT**: Primary resistance marker detected (e.g., pfcrt K76T for chloroquine)
- **PARTIAL**: Secondary/modifying mutations only
- **SENSITIVE**: No resistance markers detected
- **UNKNOWN**: Insufficient data for classification

In [ ]:
# Display drug database
print("💊 Antimalarial Drug Resistance Database:")
print("="*80)

for drug_id, drug_info in DRUG_DATABASE.items():
    print(f"\n🔹 {drug_info['name']} ({drug_id})")
    print(f"   Primary gene: {drug_info['primary_gene']}")
    print(f"   Key markers:  {', '.join(drug_info['markers'][:3])}")
    print(f"   Class: {drug_info['drug_class'].value}")

print("\n" + "="*80)

In [ ]:
# Detect drug resistance
print("🔬 Analyzing drug resistance profile...\n")

detector = ResistanceDetector()
resistance_profile = detector.analyze_vcf(
    vcf_path=str(annotated_vcf),
    sample_id=SAMPLE_ID
)

print("="*70)
print("         ANTIMALARIAL DRUG RESISTANCE REPORT")
print(f"         Sample: {SAMPLE_ID}")
print(f"         Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("="*70)

# Summary
print(f"\n📊 SUMMARY")
print(f"   Total mutations detected:  {len(resistance_profile.mutations)}")
print(f"   Resistance status:         {'RESISTANT' if resistance_profile.has_any_resistance else 'SENSITIVE'}")
if resistance_profile.resistant_drugs:
    print(f"   Drugs affected:            {', '.join(resistance_profile.resistant_drugs)}")

# Detailed drug-by-drug results
print(f"\n💊 DRUG-BY-DRUG ANALYSIS")
print("-"*70)

for result in resistance_profile.drug_results:
    status_emoji = {
        'RESISTANT': '🔴',
        'PARTIAL': '🟡', 
        'SENSITIVE': '🟢',
        'UNKNOWN': '⚪'
    }.get(result.status.name, '⚪')
    
    print(f"\n{status_emoji} {result.drug}")
    print(f"   Status:     {result.status.name}")
    print(f"   Confidence: {result.confidence:.0%}")
    if result.mutations_found:
        print(f"   Mutations:  {', '.join(result.mutations_found)}")
    if result.notes:
        print(f"   Notes:      {result.notes}")

print("\n" + "="*70)

In [ ]:
# Generate and display full resistance report
report_text = detector.generate_report(resistance_profile)

# Save report
report_file = REPORTS_DIR / f'{SAMPLE_ID}_resistance_report.txt'
with open(report_file, 'w') as f:
    f.write(report_text)

print(f"\n✅ Full report saved to: {report_file}")

In [ ]:
# Visualize resistance results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Drug resistance status summary
ax1 = axes[0]
status_counts = {'Resistant': 0, 'Partial': 0, 'Sensitive': 0}
for result in resistance_profile.drug_results:
    if result.status.name == 'RESISTANT':
        status_counts['Resistant'] += 1
    elif result.status.name == 'PARTIAL':
        status_counts['Partial'] += 1
    else:
        status_counts['Sensitive'] += 1

colors = ['#e74c3c', '#f39c12', '#27ae60']
wedges, texts, autotexts = ax1.pie(
    status_counts.values(), 
    labels=status_counts.keys(),
    colors=colors,
    autopct='%1.0f%%',
    startangle=90,
    explode=(0.05, 0.02, 0)
)
ax1.set_title(f'Drug Resistance Status\n(n={len(resistance_profile.drug_results)} drugs tested)', fontsize=12)

# 2. Per-drug resistance heatmap-style bar
ax2 = axes[1]
drugs = [r.drug for r in resistance_profile.drug_results]
status_values = []
status_colors = []
for r in resistance_profile.drug_results:
    if r.status.name == 'RESISTANT':
        status_values.append(3)
        status_colors.append('#e74c3c')
    elif r.status.name == 'PARTIAL':
        status_values.append(2)
        status_colors.append('#f39c12')
    else:
        status_values.append(1)
        status_colors.append('#27ae60')

bars = ax2.barh(drugs, status_values, color=status_colors, edgecolor='black', alpha=0.8)
ax2.set_xlim(0, 4)
ax2.set_xticks([1, 2, 3])
ax2.set_xticklabels(['Sensitive', 'Partial', 'Resistant'])
ax2.set_title('Resistance Profile by Drug', fontsize=12)
ax2.invert_yaxis()

# 3. Mutations per gene
ax3 = axes[2]
gene_mutations = {}
for mut in resistance_profile.mutations:
    gene = mut.gene
    if gene not in gene_mutations:
        gene_mutations[gene] = 0
    gene_mutations[gene] += 1

if gene_mutations:
    genes = list(gene_mutations.keys())
    mut_counts = list(gene_mutations.values())
    colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(genes)))
    bars = ax3.bar(genes, mut_counts, color=colors, edgecolor='black', alpha=0.8)
    ax3.set_ylabel('Number of Resistance Mutations')
    ax3.set_title('Resistance Mutations by Gene', fontsize=12)
    ax3.tick_params(axis='x', rotation=45)
    for bar, count in zip(bars, mut_counts):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                 str(count), ha='center', va='bottom', fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'No resistance\nmutations detected', ha='center', va='center', 
             fontsize=14, transform=ax3.transAxes)
    ax3.set_title('Resistance Mutations by Gene', fontsize=12)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'resistance_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Resistance plot saved to: {REPORTS_DIR / 'resistance_profile.png'}")

---

## 💾 Step 6: Database Storage

### Background
For molecular surveillance programs, results must be stored in a database for:
- Longitudinal tracking of resistance patterns
- Geographic analysis of resistance spread
- Aggregated reporting to health authorities
- Integration with national malaria information systems

In [ ]:
# Initialize database
db_path = DATABASE_DIR / 'malaria_surveillance.db'

print("💾 Initializing surveillance database...\n")

db_manager = DatabaseManager(str(db_path))
db_manager.initialize_database()

print(f"✅ Database created: {db_path}")
print("\n📋 Database Schema:")
print("   • samples          - Sample metadata")
print("   • variants         - Called variants")
print("   • annotations      - Functional annotations")
print("   • resistance_calls - Drug resistance results")
print("   • gene_coverage    - Per-gene sequencing metrics")

In [ ]:
# Store sample and results in database
print("📥 Storing results in database...\n")

# Insert sample metadata
sample_db_id = db_manager.insert_sample(
    sample_id=SAMPLE_METADATA['sample_id'],
    collection_date=SAMPLE_METADATA['collection_date'],
    location=SAMPLE_METADATA['location'],
    country=SAMPLE_METADATA['country'],
    region=SAMPLE_METADATA['region'],
    patient_age=SAMPLE_METADATA['patient_age'],
    parasitemia=SAMPLE_METADATA['parasitemia'],
    sequencing_platform=SAMPLE_METADATA['sequencing_platform'],
    coverage=variant_stats.mean_depth
)
print(f"   ✓ Sample metadata stored (ID: {sample_db_id})")

# Import variants from VCF
variant_count = db_manager.import_vcf(str(annotated_vcf), SAMPLE_ID)
print(f"   ✓ {variant_count} variants imported")

# Store resistance profile
db_manager.store_resistance_profile(resistance_profile)
print(f"   ✓ Resistance profile stored ({len(resistance_profile.drug_results)} drug classifications)")

print("\n✅ All results stored in database!")

In [ ]:
# Query database for statistics
print("📊 Database Statistics:\n")

stats = db_manager.get_statistics()

print(f"   Total samples:           {stats['total_samples']}")
print(f"   Total variants:          {stats['total_variants']}")
print(f"   Resistance mutations:    {stats['resistance_mutations']}")
print(f"   Countries represented:   {stats['countries']}")

In [ ]:
# Query resistance summary
print("\n💊 Resistance Summary from Database:\n")

resistance_summary = db_manager.get_resistance_summary()

if resistance_summary:
    summary_df = pd.DataFrame(resistance_summary)
    print(summary_df.to_string(index=False))
else:
    print("   No resistance data in database yet.")

In [ ]:
# Query mutation frequencies
print("\n🧬 Mutation Frequency Analysis:\n")

mutation_freqs = db_manager.get_mutation_frequencies()

if mutation_freqs:
    freq_df = pd.DataFrame(mutation_freqs)
    freq_df = freq_df.sort_values('count', ascending=False)
    print(freq_df.head(10).to_string(index=False))
else:
    print("   No mutation data in database yet.")

---

## 📈 Step 7: Surveillance Visualization

### Background
Effective molecular surveillance requires clear visualization of resistance patterns for:
- National malaria control programs
- WHO reporting
- Treatment guideline updates
- Outbreak detection

In [ ]:
# Create comprehensive surveillance dashboard
fig = plt.figure(figsize=(18, 12))

# Title
fig.suptitle('Plasmodium falciparum Molecular Surveillance Dashboard\n' + 
             f'Sample: {SAMPLE_ID} | Location: {SAMPLE_METADATA["location"]} | Date: {SAMPLE_METADATA["collection_date"]}',
             fontsize=14, fontweight='bold', y=0.98)

# Create grid for subplots
gs = fig.add_gridspec(3, 4, hspace=0.35, wspace=0.3)

# 1. Overall resistance status (large)
ax1 = fig.add_subplot(gs[0, :2])
status_data = {'Resistant': 0, 'Partial': 0, 'Sensitive': 0}
for result in resistance_profile.drug_results:
    if result.status.name == 'RESISTANT':
        status_data['Resistant'] += 1
    elif result.status.name == 'PARTIAL':
        status_data['Partial'] += 1
    else:
        status_data['Sensitive'] += 1

colors = ['#c0392b', '#f39c12', '#27ae60']
wedges, texts, autotexts = ax1.pie(
    status_data.values(), labels=status_data.keys(), colors=colors,
    autopct=lambda pct: f'{pct:.0f}%\n({int(pct/100*sum(status_data.values()))})',
    startangle=90, explode=(0.05, 0.02, 0), textprops={'fontsize': 11}
)
ax1.set_title('Overall Drug Resistance Profile', fontsize=12, fontweight='bold')

# 2. Drug-specific resistance bar chart
ax2 = fig.add_subplot(gs[0, 2:])
drug_names = [r.drug for r in resistance_profile.drug_results]
confidences = [r.confidence * 100 for r in resistance_profile.drug_results]
bar_colors = []
for r in resistance_profile.drug_results:
    if r.status.name == 'RESISTANT':
        bar_colors.append('#c0392b')
    elif r.status.name == 'PARTIAL':
        bar_colors.append('#f39c12')
    else:
        bar_colors.append('#27ae60')

bars = ax2.barh(drug_names, confidences, color=bar_colors, edgecolor='black', alpha=0.8)
ax2.set_xlabel('Classification Confidence (%)')
ax2.set_xlim(0, 110)
ax2.set_title('Drug Resistance Classification', fontsize=12, fontweight='bold')
ax2.invert_yaxis()
for bar, conf in zip(bars, confidences):
    ax2.text(conf + 2, bar.get_y() + bar.get_height()/2, f'{conf:.0f}%', 
             va='center', fontsize=9)

# 3. Mutations per gene
ax3 = fig.add_subplot(gs[1, :2])
gene_mut_counts = {}
for mut in resistance_profile.mutations:
    gene = mut.gene
    gene_mut_counts[gene] = gene_mut_counts.get(gene, 0) + 1

if gene_mut_counts:
    genes = list(gene_mut_counts.keys())
    counts = list(gene_mut_counts.values())
    colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(genes)))
    bars = ax3.bar(genes, counts, color=colors, edgecolor='black')
    ax3.set_ylabel('Mutations')
    ax3.set_title('Resistance Mutations by Gene', fontsize=12, fontweight='bold')
    for bar, count in zip(bars, counts):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                 str(count), ha='center', fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'No resistance mutations', ha='center', va='center', fontsize=12)
    ax3.set_title('Resistance Mutations by Gene', fontsize=12, fontweight='bold')

# 4. Quality metrics summary
ax4 = fig.add_subplot(gs[1, 2:])
metrics = ['Read Quality\n(Q30%)', 'Mapping Rate\n(%)', 'Mean Coverage\n(X)', 'Variants\nCalled']
values = [qc_stats.q30_percentage, alignment_stats.mapping_rate, 
          variant_stats.mean_depth, variant_stats.total_variants]
colors = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22']

# Normalize for display
normalized = [min(v/100*100, 100) if i < 2 else min(v/50*100, 100) for i, v in enumerate(values)]
bars = ax4.bar(metrics, normalized, color=colors, edgecolor='black', alpha=0.8)
ax4.set_ylabel('Relative Score')
ax4.set_ylim(0, 120)
ax4.set_title('Sample Quality Metrics', fontsize=12, fontweight='bold')
for bar, val in zip(bars, values):
    label = f'{val:.1f}' if isinstance(val, float) else str(val)
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             label, ha='center', fontweight='bold')

# 5. Mutation details table
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')

if resistance_profile.mutations:
    table_data = []
    for mut in resistance_profile.mutations[:8]:  # Show top 8
        table_data.append([
            mut.gene,
            mut.mutation,
            f'{mut.allele_frequency:.2f}',
            mut.genotype,
            f'{mut.confidence:.0%}'
        ])
    
    columns = ['Gene', 'Mutation', 'Allele Freq', 'Genotype', 'Confidence']
    table = ax5.table(
        cellText=table_data,
        colLabels=columns,
        loc='center',
        cellLoc='center',
        colColours=['#3498db']*5
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    # Style header
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(fontweight='bold', color='white')
        cell.set_edgecolor('gray')

ax5.set_title('Detected Resistance Mutations', fontsize=12, fontweight='bold', pad=20)

plt.savefig(REPORTS_DIR / 'surveillance_dashboard.png', dpi=150, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()

print(f"\n📊 Dashboard saved to: {REPORTS_DIR / 'surveillance_dashboard.png'}")

---

## 📋 Summary & Conclusions

This notebook demonstrated a complete molecular surveillance workflow for detecting antimalarial drug resistance in *Plasmodium falciparum*.

In [ ]:
# Generate final summary
print("="*70)
print("              PIPELINE EXECUTION SUMMARY")
print("="*70)

print(f"\n📁 Sample Information:")
print(f"   Sample ID:    {SAMPLE_ID}")
print(f"   Location:     {SAMPLE_METADATA['location']}, {SAMPLE_METADATA['country']}")
print(f"   Collection:   {SAMPLE_METADATA['collection_date']}")

print(f"\n📊 Processing Results:")
print(f"   Raw reads:           {qc_stats.total_reads:,}")
print(f"   Filtered reads:      {filter_stats['reads_after']:,}")
print(f"   Mapped reads:        {alignment_stats.mapped_reads:,} ({alignment_stats.mapping_rate:.1f}%)")
print(f"   Variants called:     {variant_stats.total_variants:,}")
print(f"   Resistance markers:  {annotation_stats.resistance_mutations:,}")

print(f"\n💊 Resistance Classification:")
resistant_drugs = [r.drug for r in resistance_profile.drug_results if r.status.name == 'RESISTANT']
sensitive_drugs = [r.drug for r in resistance_profile.drug_results if r.status.name == 'SENSITIVE']

if resistant_drugs:
    print(f"   🔴 RESISTANT to: {', '.join(resistant_drugs)}")
if sensitive_drugs:
    print(f"   🟢 SENSITIVE to: {', '.join(sensitive_drugs)}")

print(f"\n📂 Output Files:")
print(f"   Filtered FASTQ:     {filtered_fastq}")
print(f"   Alignment (SAM):    {sam_file}")
print(f"   Variants (VCF):     {vcf_file}")
print(f"   Annotated VCF:      {annotated_vcf}")
print(f"   Resistance report:  {report_file}")
print(f"   Database:           {db_path}")

print(f"\n📊 Visualizations:")
for img in REPORTS_DIR.glob('*.png'):
    print(f"   {img.name}")

print("\n" + "="*70)
print("✅ Pipeline completed successfully!")
print("="*70)

---

## 🔬 Next Steps for Laboratory Scientists

### Recommended Actions Based on Results:

1. **If resistance detected:**
   - Report to national malaria control program
   - Consider treatment guideline review
   - Investigate geographic spread

2. **For surveillance programs:**
   - Batch process multiple samples
   - Generate monthly/quarterly reports
   - Track temporal trends

3. **Quality improvement:**
   - Validate results with Sanger sequencing for key mutations
   - Participate in external quality assessment programs
   - Maintain reference strain controls

### Data Sources:
- **Pf3k Project**: https://www.malariagen.net/pf3k
- **PlasmoDB**: https://plasmodb.org/
- **WHO Resistance Markers**: WHO Malaria Policy Advisory Group

---

*Pipeline developed for molecular surveillance laboratories*  
*For questions or support, contact your bioinformatics team*